In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
import torch
from torch.utils.data import DataLoader

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from src.trainer import SimpleTrainer, SIBufferTrainer
from src.data_utils import (
    get_mnist_tasks,
    _extract_targets,
    get_context_sets,
    create_holdout_set,
)
from src.utils import InContextHead
from src import models
from src.buffer import MultiTaskBuffer

In [4]:
from src.trainer import IntervalTrainer, BaseTrainer
from typing import Callable
import torch.nn as nn
from src.utils import set_random_seed
from torch.utils.data import TensorDataset
import copy
import abstract_gradient_training.interval_arithmetic as interval_arithmetic
from abstract_gradient_training.bounded_models import IntervalBoundedModel
from tqdm import tqdm

In [5]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[3, 4], [8, 9], [6, 7], [0, 5], [1, 2]]


In [6]:
class ModelWrapper(nn.Module):
    def __init__(self, model, epsilon_weights=None, epsilon_bias=None):
        super().__init__()
        self.device = next(model.parameters()).device
        self.model = copy.deepcopy(model)
        self.weight_u = [
            nn.Parameter(torch.randn_like(m.weight)).to(self.device)
            for m in model
            if self.valid_layer(m)
        ]
        self.bias_u = [
            nn.Parameter(torch.randn_like(m.bias)).to(self.device)
            for m in model
            if self.valid_layer(m)
        ]
        if not epsilon_weights:
            epsilon_weights = [
                nn.Parameter(torch.ones_like(m.weight)).to(self.device)
                for m in model
                if self.valid_layer(m)
            ]
        if not epsilon_bias:
            epsilon_bias = [
                nn.Parameter(torch.ones_like(m.bias)).to(self.device)
                for m in model
                if self.valid_layer(m)
            ]

        self.epsilon_weight = [
            nn.Parameter(param.clone().detach()) for param in epsilon_weights
        ]
        self.epsilon_bias = [
            nn.Parameter(param.clone().detach()) for param in epsilon_bias
        ]

    def valid_layer(self, layer):
        return isinstance(layer, torch.nn.Linear) or isinstance(layer, torch.nn.Conv2d)

    def forward(self, x):
        i, j = 0, 0
        while i < len(self.model):
            og = self.model[i]
            i += 1
            if not self.valid_layer(og):
                x = og(x)
                continue
            combined_weight = (
                og.weight
                + torch.nn.functional.tanh(self.weight_u[j]) * self.epsilon_weight[j]
            )
            combined_bias = (
                og.bias
                + torch.nn.functional.tanh(self.bias_u[j]) * self.epsilon_bias[j]
            )
            j += 1

            if isinstance(og, torch.nn.Linear):
                x = torch.nn.functional.linear(x, combined_weight, combined_bias)
            elif isinstance(og, torch.nn.Conv2d):
                x = torch.nn.functional.conv2d(
                    x,
                    combined_weight,
                    combined_bias,
                    stride=og.stride,
                    padding=og.padding,
                )
        return x

    def extract_weights(self):
        out = []
        for weight_u, bias_u, epsilon_weight, epsilon_bias, layer in zip(
            self.weight_u,
            self.bias_u,
            self.epsilon_weight,
            self.epsilon_bias,
            [layer for layer in self.model if self.valid_layer(layer)],
        ):
            combined_weight = (
                layer.weight + torch.nn.functional.tanh(weight_u) * epsilon_weight
            )
            combined_bias = layer.bias + torch.nn.functional.tanh(bias_u) * epsilon_bias
            out.append((combined_weight, combined_bias))

        return out

    def result(self):
        params = self.extract_weights()
        i = 0
        for layer in self.model:
            if self.valid_layer(layer):
                weights, bias = params[i]
                layer.weight = nn.Parameter(weights)
                layer.bias = nn.Parameter(bias)
                i += 1

        return copy.deepcopy(self.model)

    def parameters(self):
        return (val for val in self.weight_u + self.bias_u)

In [7]:
class DualModel(nn.Module):
    def __init__(self, model, prev_params, epsilons=None, default_epsilon=0.1):
        super().__init__()
        set_random_seed(42)
        self.device = next(model.parameters()).device
        self.model = copy.deepcopy(model)
        self.prev_params = prev_params
        self.vs = list(
            zip(
                [
                    nn.Parameter(torch.randn_like(m.weight)).to(self.device)
                    for m in model
                    if self.valid_layer(m)
                ],
                [
                    nn.Parameter(torch.randn_like(m.bias)).to(self.device)
                    for m in model
                    if self.valid_layer(m)
                ],
            )
        )
        if not epsilons:
            self.epsilons = list(
                zip(
                    [
                        nn.Parameter(torch.full_like(m.weight, default_epsilon)).to(
                            self.device
                        )
                        for m in model
                        if self.valid_layer(m)
                    ],
                    [
                        nn.Parameter(torch.full_like(m.bias, default_epsilon)).to(
                            self.device
                        )
                        for m in model
                        if self.valid_layer(m)
                    ],
                )
            )
        else:
            self.epsilons = [
                [w.clone().detach(), b.clone().detach()] for w, b in epsilons
            ]

    def valid_layer(self, layer):
        return isinstance(layer, torch.nn.Linear) or isinstance(layer, torch.nn.Conv2d)

    def extract_epsilons(self):
        w_epsilons, b_epsilons = [], []
        for (w_eps_ast, b_eps_ast), (w_ast, b_ast), layer, (w_v, b_v) in zip(
            self.epsilons,
            self.prev_params,
            [layer for layer in self.model if self.valid_layer(layer)],
            self.vs,
        ):
            w_middle, b_middle = layer.weight, layer.bias
            w_inter = torch.min(
                (w_ast + w_eps_ast) - w_middle, w_middle - (w_ast - w_eps_ast)
            )
            w_eps = torch.nn.functional.sigmoid(w_v) * w_inter

            b_inter = torch.min(
                (b_ast + b_eps_ast) - b_middle, b_middle - (b_ast - b_eps_ast)
            )
            b_eps = torch.nn.functional.sigmoid(b_v) * b_inter

            w_epsilons.append(w_eps)
            b_epsilons.append(b_eps)

        return list(zip(w_epsilons, b_epsilons))

    def forward(self, x):
        i, j = 0, 0
        upper = x
        lower = x
        while i < len(self.model):
            og = self.model[i]
            i += 1
            if not self.valid_layer(og):
                upper = og(upper)
                lower = og(lower)
                continue
            w_eps_ast, b_eps_ast = self.epsilons[j]
            w_ast, b_ast = self.prev_params[j]
            w_middle, b_middle = og.weight, og.bias
            w_v, b_v = self.vs[j]

            w_inter = torch.min(
                (w_ast + w_eps_ast) - w_middle, w_middle - (w_ast - w_eps_ast)
            )
            w_eps = torch.nn.functional.sigmoid(w_v) * w_inter
            w_lower = w_middle - w_eps
            w_upper = w_middle - w_eps
            w_lower_pos = w_lower.clamp(min=0)
            w_lower_neg = w_lower.clamp(max=0)
            w_upper_pos = w_upper.clamp(min=0)
            w_upper_neg = w_upper.clamp(max=0)
            b_inter = torch.min(
                (b_ast + b_eps_ast) - b_middle, b_middle - (b_ast - b_eps_ast)
            )
            b_eps = torch.nn.functional.sigmoid(b_v) * b_inter
            b_lower = b_middle - b_eps
            b_upper = b_middle + b_eps
            j += 1

            if isinstance(og, torch.nn.Linear):
                W_l, W_u = w_middle - w_eps, w_middle + w_eps
                b_l, b_u = b_middle - b_eps, b_middle + b_eps
                lower, upper = interval_arithmetic.propagate_matmul(
                    lower, upper, W_l.T, W_u.T
                )
                lower, upper = lower + b_l, upper + b_u
                # lower_out = lower @ w_lower_pos.t() + upper @ w_lower_neg.t() + b_lower
                # upper_out = upper @ w_upper_pos.t() + lower @ w_upper_neg.t() + b_upper
                # lower = lower_out
                # upper = upper_out
            elif isinstance(og, torch.nn.Conv2d):
                W_l, W_u = w_middle - w_eps, w_middle + w_eps
                b_l, b_u = b_middle - b_eps, b_middle + b_eps
                lower, upper = interval_arithmetic.propagate_conv2d(
                    lower,
                    upper,
                    W_l,
                    W_u,
                    b_l,
                    b_u,
                    stride=og.stride,
                    padding=og.padding,
                )
                # l_lp = torch.nn.functional.conv2d(
                #     lower, w_lower_pos, None, og.stride, og.padding
                # )
                # u_ln = torch.nn.functional.conv2d(
                #     upper, w_lower_neg, None, og.stride, og.padding
                # )
                # u_up = torch.nn.functional.conv2d(
                #     upper, w_upper_pos, None, og.stride, og.padding
                # )
                # l_un = torch.nn.functional.conv2d(
                #     lower, w_upper_neg, None, og.stride, og.padding
                # )
                # lower = l_lp + u_ln
                # upper = u_up + l_un
                # lower += b_lower.view(1, b_lower.size(0), 1, 1)
                # upper += b_upper.view(1, b_upper.size(0), 1, 1)

        return torch.stack((lower, upper), dim=2)


In [8]:
def max_loss(y_pred: torch.Tensor, y_true: torch.Tensor):
    y_true = y_true.squeeze(dim=-1)
    y_true_one_hot = torch.nn.functional.one_hot(
        y_true, num_classes=y_pred.shape[1]
    ).float()
    pred_l, pred_u = torch.unbind(y_pred, dim=-1)
    min_log = y_true_one_hot * pred_l  # get lower bound prediction for target
    min_log += (
        torch.ones_like(y_true_one_hot) - y_true_one_hot
    ) * pred_u  # get upper bound prediction for non targets
    return torch.nn.functional.cross_entropy(min_log, y_true)


def min_acc(y_true: torch.Tensor, y_pred: torch.Tensor):
    y_true = y_true.squeeze(dim=-1)
    y_true_one_hot = torch.nn.functional.one_hot(
        y_true, num_classes=y_pred.shape[1]
    ).float()
    pred_l, pred_u = torch.unbind(y_pred, dim=-1)
    min_log = y_true_one_hot * pred_l  # get lower bound prediction for target
    min_log += (
        torch.ones_like(y_true_one_hot) - y_true_one_hot
    ) * pred_u  # get upper bound prediction for non targets
    return torch.sum(torch.argmax(min_log, dim=1) == y_true) / y_pred.shape[0]

In [11]:
model = models.get_mnist_model(device="cuda")

trainer = SimpleTrainer(model, seed=42)
trainer.train(train_tasks[0], val_tasks[0], epochs=1, batch_size=256)
trainer.test(test_tasks[0:2])

model = trainer.model
epsilons = None
for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    # Fit epsilon bounds
    dual = DualModel(
        model,
        [
            (
                list(model.parameters())[i].clone().detach(),
                list(model.parameters())[i + 1].clone().detach(),
            )
            for i in range(0, len(list(model.parameters())), 2)
        ],
        epsilons=epsilons,
        default_epsilon=0.1,
    )
    loader = DataLoader(
        train_tasks[i - 1],
        batch_size=64,
        shuffle=True,
        generator=torch.Generator().manual_seed(42),
    )
    criterion = max_loss
    optimizer = torch.optim.AdamW(
        [w for w, _ in dual.vs] + [b for _, b in dual.vs], lr=0.002
    )

    target_acc = 0.7
    end = False
    for epoch in (pbar := tqdm(range(100))):
        for x, y in loader:
            x, y = x.to(dual.device), y.to(dual.device)
            optimizer.zero_grad()
            out = dual(x)
            loss = criterion(out, y)
            acc = min_acc(y, out)
            if acc >= target_acc:
                end = True
                break
            loss.backward()
            optimizer.step()
        pbar.set_postfix({"max loss": f"{loss.item():.4f}", "min acc": f"{acc.item():.4}"})
        if end:
            break

    epsilons = dual.extract_epsilons()
    wrapper = ModelWrapper(
        dual.model, [w for w, _ in epsilons], [b for _, b in epsilons]
    )
    loader = DataLoader(
        train,
        batch_size=64,
        shuffle=True,
        generator=torch.Generator().manual_seed(42),
    )
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(wrapper.parameters(), lr=0.001)

    wrapper.model.train()
    for epoch in (pbar := tqdm(range(20))):
        for x, y in loader:
            x, y = x.to(wrapper.device), y.to(wrapper.device)
            optimizer.zero_grad()
            out = wrapper(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    SimpleTrainer(model).test(test_tasks[0 : i + 1])

    model = wrapper.result()

    results = SimpleTrainer(model).test(test_tasks[0 : i + 1])
    if not results[-1][1]:
        print("Catastrophic forgetting occurred.")
        break

Training Epochs: 100%|████████████████████████████████████████| 1/1 [00:05<00:00,  6.00s/it, val_loss=0.196, val_acc=0.969]


Test Results: [(0.1787, 0.9839), (14.9638, 0.0)]


100%|████████████████████████████████████████████████████████████████████████| 20/20 [00:20<00:00,  1.02s/it, loss=12.9507]


Test Results: [(0.1787, 0.9839), (14.9638, 0.0)]
Test Results: [(0.1887, 0.9839), (13.7744, 0.0)]
Catastrophic forgetting occurred.
